# Phase 1: data structure and loader check

**CRISP-DM stage:** Data understanding

**Runs on:** Kaggle, with the GPU switched off, because no neural network runs here. The private dataset `revathyh/mvtec-ad2-sheet-metal-vial` is attached, so nothing is downloaded from MVTec.

**Questions it answers:** which folders hold the training, validation and test images, where are the masks, and can anomalib read these files?

## 1. Find the data

We search for the folders instead of typing a fixed path. anomalib downloads the dataset if it cannot find `root/category`, and its download link no longer works, so a wrong path must stop the notebook here.

In [ ]:
from pathlib import Path

INPUT_FOLDER = Path("/kaggle/input")
PRODUCT_NAMES = ["sheet_metal", "vial"]

# Maps each product name to its root, the folder that contains the product folder
product_roots = {}

for train_folder in INPUT_FOLDER.rglob("train"):
    product_folder = train_folder.parent
    if product_folder.name in PRODUCT_NAMES:
        product_roots[product_folder.name] = product_folder.parent

for product_name in PRODUCT_NAMES:
    if product_name not in product_roots:
        # Stop here, so anomalib never tries its download
        raise FileNotFoundError("Could not find the folder for " + product_name)
    print(product_name, "root:", product_roots[product_name])

## 2. Folder structure

We count the files in each folder. Printing every file name would mean about 1,700 lines, and the counts answer the question.

In [ ]:
# Collected in a list so the same lines can be printed and saved to a file
report_lines = []

for product_name in PRODUCT_NAMES:
    root = product_roots[product_name]
    report_lines.append("")
    report_lines.append("Folders for " + product_name + " (file count, then folder)")

    # rglob("*") finds everything at any depth below the product folder
    for path in sorted((root / product_name).rglob("*")):
        if path.is_dir():
            file_count = 0
            for item in path.iterdir():
                if item.is_file():
                    file_count = file_count + 1
            # Folders that only hold other folders are skipped to keep the list short
            if file_count > 0:
                report_lines.append(str(file_count) + "  " + str(path.relative_to(root)))

for line in report_lines:
    print(line)

## 3. Install anomalib

The exact version is pinned so that a later release cannot change how the data is read.

In [ ]:
!pip install anomalib==2.7.0

## 4. Test the loader

`MVTecAD2` is anomalib's object for this dataset.

- `root` is the folder containing the product folder, and `category` is the product folder's name.
- `num_workers=0` loads images in the main process. The default is 8 helper processes, which this small test does not need.
- We rely on three defaults: batch sizes of 32, which have no effect because no batches are made here; the public test set; and no augmentation.

`prepare_data()` downloads only if `root/category` is missing. `setup()` builds the five sets of images.

In [ ]:
import platform

import anomalib
import torch
from anomalib.data import MVTecAD2

report_lines.append("")
report_lines.append("python: " + platform.python_version())
report_lines.append("anomalib: " + anomalib.__version__)
report_lines.append("pytorch: " + torch.__version__)
report_lines.append("gpu available: " + str(torch.cuda.is_available()))

datamodules = {}

for product_name in PRODUCT_NAMES:
    datamodule = MVTecAD2(
        root=product_roots[product_name],
        category=product_name,
        num_workers=0,
    )
    datamodule.prepare_data()
    datamodule.setup()
    datamodules[product_name] = datamodule

    report_lines.append("")
    report_lines.append("Images the loader found for " + product_name)
    report_lines.append("train: " + str(len(datamodule.train_data)))
    report_lines.append("validation: " + str(len(datamodule.val_data)))
    report_lines.append("public test: " + str(len(datamodule.test_public_data)))
    report_lines.append("private test: " + str(len(datamodule.test_private_data)))
    report_lines.append("private mixed test: " + str(len(datamodule.test_private_mixed_data)))

for line in report_lines:
    print(line)

## 5. Open one sample

We read one training image, and one defective test image with its mask. No resizing is set up, so the shapes should be the original image sizes. The full checks on every file come later.

In [ ]:
for product_name in PRODUCT_NAMES:
    datamodule = datamodules[product_name]
    report_lines.append("")
    report_lines.append("One sample from " + product_name)

    train_item = datamodule.train_data[0]
    report_lines.append("train image shape: " + str(list(train_item.image.shape)))

    # label_index is 1 for defective images, so this picks the first defective one
    samples = datamodule.test_public_data.samples
    defective_samples = samples[samples["label_index"] == 1]
    first_defective = defective_samples.index[0]

    test_item = datamodule.test_public_data[first_defective]
    report_lines.append("defective image file: " + Path(test_item.image_path).name)
    report_lines.append("defective image shape: " + str(list(test_item.image.shape)))
    report_lines.append("mask file: " + Path(test_item.mask_path).name)
    report_lines.append("mask shape: " + str(list(test_item.gt_mask.shape)))
    # A mask should only say defect or no defect, so it should hold two values at most
    report_lines.append("mask values: " + str(torch.unique(test_item.gt_mask.int()).tolist()))

for line in report_lines:
    print(line)

## 6. Check nothing downloaded, then save

anomalib's download would leave an archive behind, so finding none in the working folder or next to the data shows it never ran. Files saved in the working folder become the notebook's output, so they survive a disconnect.

In [ ]:
downloaded_archives = list(Path.cwd().rglob("*.tar.gz"))

for product_name in PRODUCT_NAMES:
    root = product_roots[product_name]
    for archive in root.glob("*.tar.gz"):
        downloaded_archives.append(archive)

report_lines.append("")
report_lines.append("archives found after loading: " + str(len(downloaded_archives)))

with open("folder_structure.txt", "w") as report_file:
    for line in report_lines:
        report_file.write(line + "\n")

print("archives found after loading:", len(downloaded_archives))
print("Saved folder_structure.txt")